In [1]:
!pip install ultralytics opencv-python numpy

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
   ----------------- ---------------------- 0.5/1.2 MB 560.1 kB/s eta 0:00:02
   ----------------- ---------------------- 0.5/1.2 MB 560.1 kB/s eta 0:00:02
   ----------------- ---------------------- 0.5/1.2 MB 560.1 kB/s eta 0:00:02
   ----------------- ---------------------- 0.5/1.2 MB 560.1 kB/s eta 0:00:02
   ------------------------- -------------- 0.8/1.2 MB 435.8 kB/s eta 0:00:02
   ------------------------- -------------- 0.8/1.2 MB 435.8 kB/s eta 0:00:02
   ---------------------------------- ----- 1.0/1

In [2]:
import cv2
import numpy as np
import math
import sqlite3
from datetime import datetime
from collections import defaultdict

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [3]:
def analyze_video_motion(video_path, sample_frames=60):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'❌ Video not open: {video_path}')
        return None

    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f'📹 Video: {width}×{height} @ {fps}fps | {total} frames ({total/fps:.1f}s)')

    flow_x_list = []
    flow_y_list = []
    prev_gray   = None

    step = max(1, total // sample_frames)

    for i in range(0, min(total - 1, sample_frames * step), step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (15, 15), 0)

        if prev_gray is not None:
            # Dense optical flow
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])

            # only meaningful motion
            motion_mask = mag > 1.5
            if motion_mask.sum() > 200:
                fx = flow[..., 0][motion_mask].mean()
                fy = flow[..., 1][motion_mask].mean()
                flow_x_list.append(fx)
                flow_y_list.append(fy)

        prev_gray = gray

    cap.release()

    if not flow_x_list:
        print('⚠️ No motion detected')
        return {'primary_direction': 'vertical', 'recommended_mode': 'HORIZONTAL',
                'flow_x': 0, 'flow_y': 0, 'angle_deg': 90,
                'width': width, 'height': height, 'fps': fps, 'total': total}

    avg_fx = np.mean(flow_x_list)
    avg_fy = np.mean(flow_y_list)
    std_fx = np.std(flow_x_list)
    std_fy = np.std(flow_y_list)

    # Motion angle (0° = right, 90° = down)
    angle_deg = math.degrees(math.atan2(abs(avg_fy), abs(avg_fx)))

    # Primary direction determine
    h_ratio = abs(avg_fx) / (abs(avg_fx) + abs(avg_fy) + 1e-6)

    if h_ratio > 0.65:
        primary   = 'horizontal'
        rec_mode  = 'VERTICAL'    # Horizontal motion → vertical line
        direction = '→ RIGHT' if avg_fx > 0 else '← LEFT'
    elif h_ratio < 0.35:
        primary   = 'vertical'
        rec_mode  = 'HORIZONTAL'  # Vertical motion → horizontal line
        direction = '↓ DOWN' if avg_fy > 0 else '↑ UP'
    else:
        primary   = 'diagonal'
        rec_mode  = 'DIAGONAL'
        direction = f'↘ DIAGONAL ({angle_deg:.0f}°)'

    result = {
        'primary_direction': primary,
        'recommended_mode':  rec_mode,
        'flow_x':    avg_fx,
        'flow_y':    avg_fy,
        'std_x':     std_fx,
        'std_y':     std_fy,
        'h_ratio':   h_ratio,
        'angle_deg': angle_deg,
        'direction': direction,
        'width':     width,
        'height':    height,
        'fps':       fps,
        'total':     total
    }

    print(f'\n🔍 Motion Analysis Results:')
    print(f'   Avg flow X      : {avg_fx:+.2f} px/frame')
    print(f'   Avg flow Y      : {avg_fy:+.2f} px/frame')
    print(f'   H/V ratio       : {h_ratio:.2f} (1.0=pure horizontal, 0.0=pure vertical)')
    print(f'   Motion angle    : {angle_deg:.1f}°')
    print(f'   Direction       : {direction}')
    print(f'\n   ✅ Recommended LINE_MODE: [{rec_mode}]')

    return result

print('✅ Analyzer function ready!')

✅ Analyzer function ready!


In [4]:
class SimpleTracker:
    def __init__(self, max_disappeared=35, iou_threshold=0.25):
        self.next_id          = 1
        self.tracks           = {}
        self.max_disappeared  = max_disappeared
        self.iou_threshold    = iou_threshold

    def _iou(self, A, B):
        xA = max(A[0], B[0]); yA = max(A[1], B[1])
        xB = min(A[2], B[2]); yB = min(A[3], B[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        if inter == 0:
            return 0.0
        areaA = (A[2]-A[0]) * (A[3]-A[1])
        areaB = (B[2]-B[0]) * (B[3]-B[1])
        return inter / float(areaA + areaB - inter)

    def _register(self, det):
        x1, y1, x2, y2, cls_name, conf = det
        self.tracks[self.next_id] = {
            'bbox'       : (x1, y1, x2, y2),
            'centroid'   : ((x1+x2)//2, (y1+y2)//2),
            'class_name' : cls_name,
            'conf'       : conf,
            'disappeared': 0,
            'counted'    : False,
            'prev_side'  : None
        }
        self.next_id += 1

    def update(self, detections):
        if not detections:
            for tid in list(self.tracks.keys()):
                self.tracks[tid]['disappeared'] += 1
                if self.tracks[tid]['disappeared'] > self.max_disappeared:
                    del self.tracks[tid]
            return []

        det_boxes = [(d[0],d[1],d[2],d[3]) for d in detections]

        if not self.tracks:
            for det in detections:
                self._register(det)
        else:
            track_ids   = list(self.tracks.keys())
            track_boxes = [self.tracks[tid]['bbox'] for tid in track_ids]

            iou_mat = np.zeros((len(track_ids), len(det_boxes)))
            for t, tb in enumerate(track_boxes):
                for d, db in enumerate(det_boxes):
                    iou_mat[t, d] = self._iou(tb, db)

            matched_t = set(); matched_d = set()

            flat_sorted = np.argsort(iou_mat.ravel())[::-1]
            for idx in flat_sorted:
                t_i = idx // len(det_boxes)
                d_i = idx  % len(det_boxes)
                if iou_mat[t_i, d_i] < self.iou_threshold:
                    break
                if t_i in matched_t or d_i in matched_d:
                    continue
                tid = track_ids[t_i]
                det = detections[d_i]
                x1,y1,x2,y2 = det[0],det[1],det[2],det[3]
                self.tracks[tid].update({
                    'bbox'       : (x1, y1, x2, y2),
                    'centroid'   : ((x1+x2)//2, (y1+y2)//2),
                    'class_name' : det[4],
                    'conf'       : det[5],
                    'disappeared': 0
                })
                matched_t.add(t_i); matched_d.add(d_i)

            # Unmatched tracks
            for t_i, tid in enumerate(track_ids):
                if t_i not in matched_t:
                    self.tracks[tid]['disappeared'] += 1
                    if self.tracks[tid]['disappeared'] > self.max_disappeared:
                        del self.tracks[tid]

            # Unmatched detections → new tracks
            for d_i, det in enumerate(detections):
                if d_i not in matched_d:
                    self._register(det)

        out = []
        for tid, info in self.tracks.items():
            x1,y1,x2,y2 = info['bbox']
            cx,cy        = info['centroid']
            out.append((tid, x1, y1, x2, y2, cx, cy, info['class_name'], info['conf']))
        return out

print("✅ SimpleTracker ready!")

✅ SimpleTracker ready!


In [5]:
def build_counting_line(motion_info, ratio=0.90):
    W   = motion_info['width']
    H   = motion_info['height']
    mode = motion_info['recommended_mode']

    if mode == 'HORIZONTAL':
        y = int(H * ratio)
        config = {
            'mode'  : 'HORIZONTAL',
            'pt1'   : (0, y),
            'pt2'   : (W, y),
            'coord' : y,
            'axis'  : 'y',
            'label' : f'COUNTING LINE  (y={y})'
        }
        print(f"📏 HORIZONTAL line at y={y}  ({ratio*100:.0f}% of height={H})")

    elif mode == 'VERTICAL':
        x = int(W * ratio)
        config = {
            'mode'  : 'VERTICAL',
            'pt1'   : (x, 0),
            'pt2'   : (x, H),
            'coord' : x,
            'axis'  : 'x',
            'label' : f'COUNTING LINE  (x={x})'
        }
        print(f"📏 VERTICAL line at x={x}  ({ratio*100:.0f}% of width={W})")

    else:  # DIAGONAL fallback
        y = int(H * ratio)
        config = {
            'mode'  : 'HORIZONTAL',
            'pt1'   : (0, y),
            'pt2'   : (W, y),
            'coord' : y,
            'axis'  : 'y',
            'label' : f'COUNTING LINE (diagonal→fallback horizontal, y={y})'
        }
        print(f"📏 DIAGONAL fallback → HORIZONTAL line at y={y}")

    print(f"   Line: {config['pt1']}  →  {config['pt2']}")
    return config

print("✅ Line builder ready!")

✅ Line builder ready!


In [6]:
def check_crossing(track_info, line_config, tolerance=8):
    if track_info['counted']:
        return False

    cx, cy = track_info['centroid']
    coord  = line_config['coord']
    axis   = line_config['axis']

    val = cy if axis == 'y' else cx

    current_side = 1 if val > coord else -1

    prev_side = track_info.get('prev_side', None)

    if prev_side is None:
        track_info['prev_side'] = current_side
        return False

    near_line = abs(val - coord) <= tolerance

    side_changed = (prev_side != current_side)

    track_info['prev_side'] = current_side

    if near_line or side_changed:
        return True

    return False

print("✅ Crossing checker ready!")

✅ Crossing checker ready!


In [7]:
def run_tracking(video_path, output_path="output_tracked.mp4", line_ratio=0.87):
    from ultralytics import YOLO # Import YOLO model
    model = YOLO('yolov8n.pt') # Load a pre-trained YOLOv8n model

    # STEP 1 — Motion Analysis
    print("=" * 55)
    print("  STEP 1 → Motion Analysis")
    print("=" * 55)
    motion_info = analyze_video_motion(video_path, sample_frames=60)
    if motion_info is None:
        return

    # STEP 2 — Counting Line
    print("\n" + "=" * 55)
    print("  STEP 2 → Counting Line")
    print("=" * 55)
    line_cfg = build_counting_line(motion_info, ratio=line_ratio)

    # STEP 3 — Video Setup
    cap = cv2.VideoCapture(video_path)
    W   = motion_info['width']
    H   = motion_info['height']
    FPS = motion_info['fps'] or 30
    TOT = motion_info['total']

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_wr = cv2.VideoWriter(output_path, fourcc, FPS, (W, H))

    CLASS_COLORS = {
        'car'       : (0,   255,   0),
        'motorcycle': (255, 165,   0),
        'bus'       : (0,   0,   255),
        'truck'     : (0,   255, 255),
    }
    classes_order = ['car', 'motorcycle', 'bus', 'truck']

    tracker     = SimpleTracker(max_disappeared=40, iou_threshold=0.25)
    count       = defaultdict(int)
    counted_ids = set()
    frame_num   = 0

    print("\n" + "=" * 55)
    print("  STEP 3 → Processing Frames...")
    print("=" * 55)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1

        if frame_num % 100 == 0:
            print(f"  ▶ Frame {frame_num}/{TOT} | Counted: {sum(count.values())}")

        # ── Perform object detection with YOLO ────────────────────────
        results = model(frame, verbose=False) # Run YOLO inference
        detections = []
        for r in results:
            for *xyxy, conf, cls in r.boxes.data:
                x1, y1, x2, y2 = map(int, xyxy)
                cls_name = model.names[int(cls)]
                detections.append((x1, y1, x2, y2, cls_name, float(conf)))

        tracked = tracker.update(detections)

        # ── Counting Line Draw ────────────────────────────────────────
        cv2.line(frame, line_cfg['pt1'], line_cfg['pt2'], (0, 255, 255), 3)

        # ── Har tracked vehicle process karo ─────────────────────────
        for (tid, x1, y1, x2, y2, cx, cy, cls_name, conf) in tracked:
            color = CLASS_COLORS.get(cls_name, (200, 200, 200))

            # ID label lao box ke upar
            label = f"ID:{tid}"
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 2)
            cv2.rectangle(frame, (x1, y1-lh-10), (x1+lw+4, y1), color, -1)
            cv2.putText(frame, label, (x1+2, y1-4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 0, 0), 2)

            # Centroid dot
            cv2.circle(frame, (cx, cy), 5, (255, 255, 255), -1)
            cv2.circle(frame, (cx, cy), 8, color, 2)

            # ── Crossing Check — sirf ek baar ────────────────────────
            if tid not in counted_ids:
                track_info = tracker.tracks.get(tid)
                if track_info and check_crossing(track_info, line_cfg, tolerance=8):
                    counted_ids.add(tid)
                    count[cls_name] += 1
                    tracker.tracks[tid]['counted'] = True
                    cv2.circle(frame, (cx, cy), 22, (0, 255, 0), 3)
                    print(f"    ✅ ID:{tid} [{cls_name}] | Total:{sum(count.values())}")

        # ── TOP COUNTER BOX ───────────────────────────────────────────
        total_count = sum(count.values())
        box_x, box_y = 10, 10
        box_w        = 260
        header_h     = 40
        row_h        = 28
        box_h        = header_h + (len(classes_order) * row_h) + 10

        overlay = frame.copy()
        cv2.rectangle(overlay, (box_x, box_y),
                      (box_x+box_w, box_y+box_h), (15, 15, 15), -1)
        cv2.addWeighted(overlay, 0.70, frame, 0.30, 0, frame)
        cv2.rectangle(frame, (box_x, box_y),
                      (box_x+box_w, box_y+box_h), (0, 255, 255), 2)

        # TOTAL row
        cv2.putText(frame, f"TOTAL : {total_count}",
                    (box_x+10, box_y+28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.70, (0, 255, 255), 2)

        cv2.line(frame,
                 (box_x+6,       box_y+header_h),
                 (box_x+box_w-6, box_y+header_h),
                 (80, 80, 80), 1)

        # Per-class rows
        for i, cls in enumerate(classes_order):
            row_y = box_y + header_h + (i * row_h) + 22
            color = CLASS_COLORS[cls]
            cv2.circle(frame, (box_x+16, row_y-6), 6, color, -1)
            cv2.putText(frame, f"{cls:<12}: {count[cls]}",
                        (box_x+30, row_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 2)

        out_wr.write(frame)

    cap.release()
    out_wr.release()

    print("\n" + "=" * 55)
    print("         ✅  COMPLETE")
    print("=" * 55)
    for cls in classes_order:
        print(f"  {cls:<13}: {count[cls]}")
    print(f"  {'TOTAL':<13}: {sum(count.values())}")
    print(f"\n💾 Output → {output_path}")

    return count, line_cfg

In [8]:
run_tracking("/videos/output_detection.mp4")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\lenovo\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  STEP 1 → Motion Analysis
❌ Video not open: /videos/output_detection.mp4
